# OECD DAC2 ODA Duplicate & Fix


In [3]:
import pandas as pd
df = pd.read_csv('../../data/processed/oecd_dac2_oda.csv')
print(f'Total rows: {len(df):,}')
print(f'Columns: {df.columns.tolist()}')
df.head()

Total rows: 104,087
Columns: ['donor', 'recipient', 'year', 'oda_value', 'a_iso3', 'b_iso3']


,donor,recipient,year,oda_value,a_iso3,b_iso3
0,Australia,Brazil,1992,0.03,AUS,BRA
1,Australia,Guatemala,1992,0.03,AUS,GTM
2,Australia,Iran,1992,0.03,AUS,IRN
3,Australia,Senegal,1992,0.03,AUS,SEN
4,Australia,Armenia,1993,0.03,AUS,ARM


## duplicate problem?

In [4]:
dupes = df[df.duplicated(subset=['a_iso3', 'b_iso3', 'year'], keep=False)]
n_groups = dupes.groupby(['a_iso3', 'b_iso3', 'year']).ngroups

print(f'Rows involved in duplicates: {len(dupes)} out of {len(df):,} ({len(dupes)/len(df)*100:.2f}%)')
print(f'Unique duplicate groups: {n_groups}')
print(f'Group sizes: {dupes.groupby(["a_iso3", "b_iso3", "year"]).size().value_counts().to_dict()}')
print(f'\n-> All duplicates are pairs aka 2 rows each), nothing else')

Rows involved in duplicates: 496 out of 104,087 (0.48%)
Unique duplicate groups: 248
Group sizes: {2: 248}

-> All duplicates are pairs aka 2 rows each), nothing else


## which recipients affected?

In [5]:
affected_recipients = dupes.groupby('b_iso3')['recipient'].first().reset_index()
affected_recipients.columns = ['ISO3', 'Recipient']
affected_recipients['Duplicate Rows'] = dupes.groupby('b_iso3').size().values

print(f'N of recips affected {len(affected_recipients)}')
print()
affected_recipients

N of recips affected 1



,ISO3,Recipient,Duplicate Rows
0,FSM,Micronesia,496


## Germany & Micronesia 2022 issue check


In [6]:
deu_fsm_2022 = df[(df.a_iso3 == 'DEU') & (df.b_iso3 == 'FSM') & (df.year == 2022)]
print('Germany → Micronesia 2022:')
print(deu_fsm_2022[['donor', 'recipient', 'year', 'oda_value']].to_string(index=False))
print(f'\n There are TWO rows, not one.')
print(f' The 0.004922 is one ODA channel; the 3.919717 is another.')
print(f' Summed total: {deu_fsm_2022.oda_value.sum():.6f} million USD')

Germany → Micronesia 2022:
  donor  recipient  year  oda_value
Germany Micronesia  2022   0.004922
Germany Micronesia  2022   3.919717

 There are TWO rows, not one.
 The 0.004922 is one ODA channel; the 3.919717 is another.
 Summed total: 3.924639 million USD


## duplicate pairs across ALL donorss

In [7]:

sample_groups = []
seen_donors = set()
for (a, b, y), grp in dupes.groupby(['a_iso3', 'b_iso3', 'year']):
    if a not in seen_donors:
        seen_donors.add(a)
        row = {
            'Donor': grp.donor.iloc[0],
            'Year': y,
            'Value 1': grp.oda_value.iloc[0],
            'Value 2': grp.oda_value.iloc[1],
            'Identical?': grp.oda_value.iloc[0] == grp.oda_value.iloc[1],
            'Sum': grp.oda_value.sum()
        }
        sample_groups.append(row)

sample_df = pd.DataFrame(sample_groups).sort_values('Sum', ascending=False).head(15)
print('Sample FSM duplicate pairs (one per donor):')
sample_df

Sample FSM duplicate pairs (one per donor):


,Donor,Year,Value 1,Value 2,Identical?,Sum
13,Japan,1992,10.300000,34.580000,False,44.880000
0,United Arab Emirates,2015,1.790000,14.730000,False,16.520000
1,Australia,1993,0.880000,4.740000,False,5.620000
5,Germany,1992,1.130000,1.140000,False,2.270000
22,Türkiye,2008,0.250000,1.880000,False,2.130000
23,United States,1992,1.000000,1.000000,True,2.000000
12,Italy,2019,0.283371,1.595865,False,1.879236
17,New Zealand,1992,1.460000,0.050000,False,1.510000
15,Luxembourg,2013,0.296691,0.336540,False,0.633231
3,Canada,2005,0.200000,0.200000,True,0.400000


In [8]:
#Some pairs have identical values (true duplicates), others have different values (sub-entries from different ODA channels). Summing handles both cases correctly.

## fix

In [9]:
df_fixed = df.groupby(['donor', 'recipient', 'year', 'a_iso3', 'b_iso3'], as_index=False)['oda_value'].sum()


remaining_dupes = df_fixed.duplicated(subset=['a_iso3', 'b_iso3', 'year']).sum()

print(f'Before: {len(df):,} rows, {dupes.groupby(["a_iso3","b_iso3","year"]).ngroups} duplicate groups')
print(f'After:  {len(df_fixed):,} rows, {remaining_dupes} duplicate groups')
print(f'Rows removed: {len(df) - len(df_fixed)}')
print(f'\n All 248 duplicates resolved. Zero remaining.')

Before: 104,087 rows, 248 duplicate groups
After:  103,839 rows, 0 duplicate groups
Rows removed: 248

 All 248 duplicates resolved. Zero remaining.


## DOUBLE CHECK ALL GOOOOOOS

In [10]:
# GERMANY MICRONESIA 
deu_fsm_fixed = df_fixed[(df_fixed.a_iso3 == 'DEU') & (df_fixed.b_iso3 == 'FSM')]
print('Germany & Micronesia (after fix):')
print(deu_fsm_fixed[['donor', 'recipient', 'year', 'oda_value']].to_string(index=False))
print(f'\n 2022 now correctly shows {deu_fsm_fixed[deu_fsm_fixed.year==2022].oda_value.values[0]:.6f} million USD (sum of both channels)')

Germany & Micronesia (after fix):
  donor  recipient  year  oda_value
Germany Micronesia  1992   2.270000
Germany Micronesia  1993   1.140000
Germany Micronesia  1994   1.350000
Germany Micronesia  1995   0.020000
Germany Micronesia  1997   0.010000
Germany Micronesia  1998   0.060000
Germany Micronesia  1999   0.020000
Germany Micronesia  2000   0.060000
Germany Micronesia  2001   0.060000
Germany Micronesia  2002   0.060000
Germany Micronesia  2003   0.100000
Germany Micronesia  2004   0.070000
Germany Micronesia  2005   0.080000
Germany Micronesia  2006   0.010000
Germany Micronesia  2007   0.010000
Germany Micronesia  2008   0.190000
Germany Micronesia  2009   0.300000
Germany Micronesia  2010   0.340000
Germany Micronesia  2011   0.360000
Germany Micronesia  2012   0.299118
Germany Micronesia  2013   0.131645
Germany Micronesia  2014   0.105693
Germany Micronesia  2015   0.164502
Germany Micronesia  2016   0.089006
Germany Micronesia  2017   0.440182
Germany Micronesia  2018   1.6

In [11]:
# Spot check
print('Top 10 ODA flows')
df_fixed.nlargest(10, 'oda_value')[['donor', 'recipient', 'year', 'oda_value']]

Top 10 ODA flows


,donor,recipient,year,oda_value
103594,United States,Ukraine,2023,11790.044351
101409,United States,Iraq,2005,11227.790000
103595,United States,Ukraine,2024,9532.068795
103593,United States,Ukraine,2022,9238.051948
92511,Türkiye,Syrian Arab Republic,2017,7246.780000
92513,Türkiye,Syrian Arab Republic,2019,7202.359492
92514,Türkiye,Syrian Arab Republic,2020,7078.175985
92512,Türkiye,Syrian Arab Republic,2018,6698.514820
92515,Türkiye,Syrian Arab Republic,2021,6689.830090
92518,Türkiye,Syrian Arab Republic,2024,6011.416127


## Save fixed version

Overwrites the processed file with duplicates resolved.

In [12]:
# Save fixed version
df_fixed.to_csv('../../data/processed/oecd_dac2_oda.csv', index=False)
print(f'Saved fixed file: {len(df_fixed):,} rows (was {len(df):,})')

Saved fixed file: 103,839 rows (was 104,087)


## Summary

- **problem:** 248 duplicate groups, ALL in Micronesia (FSM), 0.48% of data
- **Cause:** Two ODA sub-entries (likely different reporting channels) collapsed into the same donor-recipient-year key
